# Tutorial: Improving GRPO Reinforcement Learning Fine-Tuning with Better Reward Structuring

This lab is adapted from **Unsloth**, and we will follow the notebook **cell by cell**. Our aim is *not* to modify the model architecture or optimizer, but to study one of the central ingredients of reinforcement learning: the **reward function**.

The core question in this tutorial is simple: **how much can model behavior change when we change only the reward construction?**

To answer this, we compare four checkpoints within the same GRPO pipeline:

1. **SFT-only baseline (no RL fine-tuning)**  
   This checkpoint is saved immediately after supervised fine-tuning and serves as our pre-RL baseline. It lets us measure what GRPO adds beyond the behavior already learned from SFT.

2. **Bad reward baseline 1: format-only reward**  
   In this setting, the model is rewarded only for producing the expected output structure. This is intentionally weak, because it can reward well-formatted but incorrect answers.

3. **Bad reward baseline 2: has-a-number reward**  
   In this setting, the model is rewarded simply for producing an answer containing a number. This is also intentionally weak, because it is easy to game and is only weakly related to solving the task.

4. **Slightly better reward baseline: binary correctness**  
   In this setting, the model receives a reward of 1 for a correct final answer and 0 otherwise. This is still simple and limited, but it is more meaningfully aligned with the task than the two weak baselines above.

## Continuing GRPO from a prepared checkpoint

GRPO training can be slow, so in this lab we support continuing from a checkpoint that was prepared in advance and saved in Google Drive.

Workflow:
- if a GRPO checkpoint already exists for the selected experiment, resume from the latest one
- run only a small number of extra steps during the lab
- save the new checkpoints back to Google Drive
- if no GRPO checkpoint exists yet, fall back to the SFT-only baseline and start a fresh GRPO run

This makes the lab faster while still letting us observe how training continues under different reward constructions.

By the end of the lab, we should see that **reward design strongly affects what the model learns**, and that simple rewards can fail in different ways. The final submission task then asks you to design a reward that improves on these provided baselines.

For more background, conceptual context, and implementation details, please refer to the accompanying lecture slides for this lab.

### Installation

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
import json
import re
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

Goal: To convert `Qwen3-4B-Base` into a reasoning model via GRPO by using OpenR1's Math dataset.

We first pre fine-tune the model to make GRPO skip trying to match formatting - this speeds GRPO up.

In [ ]:
from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer
from transformers import AutoTokenizer

# Model specs:
BASE_MODEL_NAME = "unsloth/Qwen3-4B-Base"
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 03-30 12:48:30 [__init__.py:244] Automatically detected platform cuda.
ERROR 03-30 12:48:35 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


## Saving checkpoints to Google Drive

Because this notebook runs in Google Colab, files saved only to the local runtime may be lost when the session disconnects or resets.

To keep the baseline and GRPO checkpoints available for later comparison, we save them to **Google Drive**.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/grpo_lab_checkpoints"
os.makedirs(DRIVE_ROOT, exist_ok=True)

Mounted at /content/drive


Helpers to load the checkpoints for further training or inference

In [ ]:
def get_latest_checkpoint(checkpoint_root: str):
    """
    Return the latest Trainer checkpoint folder inside checkpoint_root,
    e.g. .../checkpoint-25, .../checkpoint-50, etc.
    """
    if not os.path.isdir(checkpoint_root):
        return None

    checkpoint_dirs = []
    for name in os.listdir(checkpoint_root):
        full_path = os.path.join(checkpoint_root, name)
        if os.path.isdir(full_path) and re.fullmatch(r"checkpoint-\d+", name):
            step = int(name.split("-")[-1])
            checkpoint_dirs.append((step, full_path))

    if not checkpoint_dirs:
        return None

    checkpoint_dirs.sort(key=lambda x: x[0])
    return checkpoint_dirs[-1][1]


def get_checkpoint_step(checkpoint_dir: str) -> int:
    """
    Read global_step from trainer_state.json if available.
    Falls back to parsing checkpoint-N from the folder name.
    """
    if checkpoint_dir is None:
        return 0

    trainer_state_path = os.path.join(checkpoint_dir, "trainer_state.json")
    if os.path.exists(trainer_state_path):
        with open(trainer_state_path, "r") as f:
            state = json.load(f)
        return int(state.get("global_step", 0))

    match = re.search(r"checkpoint-(\d+)$", checkpoint_dir)
    if match is not None:
        return int(match.group(1))

    return 0

## Optional: copy prepared checkpoints from a shared Google Drive folder

If a shared folder containing prepared checkpoints is available, you can copy those checkpoints into your own lab folder in Google Drive.

### One-time setup
Before running the copy cell:

1. Open the shared Google Drive folder link [https://drive.google.com/drive/folders/14fBAB6OiJH01FOPI2UdDNXI4zsqZSdTd?usp=sharing](https://drive.google.com/drive/folders/14fBAB6OiJH01FOPI2UdDNXI4zsqZSdTd?usp=sharing) provided by the instructor
2. In Google Drive, add a **shortcut** to that folder
3. Place the shortcut inside **My Drive**
4. Put it inside a folder named **Shared checkpoints** so the path is easy to use

After that, the shared folder should appear in your Drive in a location like:

`My Drive / Shared checkpoints / grpo_lab_checkpoints`

In Colab, this corresponds to a mounted path such as:

`/content/drive/MyDrive/Shared checkpoints/grpo_lab_checkpoints`

The notebook uses that mounted path as `SHARED_DIR`, and then copies the checkpoint folders into your own DRIVE_ROOT lab folder.

### What happens next
- If the shared folder exists and contains the expected checkpoints, they will be copied into your Drive
- If the shared folder does not exist, or the checkpoints are not found, the notebook will continue with an empty checkpoint directory for now

This allows the lab to work in both cases:
1. resuming from prepared checkpoints when available
2. starting fresh when they are not

In [ ]:
import shutil

SHARED_DIR = "/content/drive/MyDrive/Shared checkpoints/grpo_lab_checkpoints"

EXPECTED_FOLDERS = [
    "sft_only_baseline",
    "grpo_better_binary_reward",
    "grpo_bad_has_number",
    "grpo_bad_format_only"
]

if os.path.exists(SHARED_DIR) and os.path.isdir(SHARED_DIR):
    print(f"Found shared folder: {SHARED_DIR}")

    available_folders = []
    missing_folders = []

    for folder_name in EXPECTED_FOLDERS:
        src = os.path.join(SHARED_DIR, folder_name)
        if os.path.exists(src) and os.path.isdir(src):
            available_folders.append(folder_name)
        else:
            missing_folders.append(folder_name)

    if available_folders:
        print("Available checkpoint folders in shared directory:")
        for folder_name in available_folders:
            print(f"  - {folder_name}")

        if missing_folders:
            print("Missing expected folders:")
            for folder_name in missing_folders:
                print(f"  - {folder_name}")

        for folder_name in available_folders:
            src = os.path.join(SHARED_DIR, folder_name)
            dst = os.path.join(DRIVE_ROOT, folder_name)

            if os.path.exists(dst):
                print(f"[SKIP] Already exists in your Drive: {dst}")
            else:
                print(f"[COPY] {src} -> {dst}")
                shutil.copytree(src, dst)

        print("Checkpoint copy step complete.")
    else:
        print("Shared folder exists, but no expected checkpoint folders were found.")
        print("Proceeding with an empty checkpoint directory for now.")
else:
    print(f"Shared folder not found: {SHARED_DIR}")
    print("Proceeding with an empty checkpoint directory for now.")

print(f"Using DRIVE_ROOT: {DRIVE_ROOT}")

Found shared folder: /content/drive/MyDrive/Shared checkpoints/grpo_lab_checkpoints
Available checkpoint folders in shared directory:
  - sft_only_baseline
  - grpo_better_binary_reward
  - grpo_bad_has_number
  - grpo_bad_format_only
[SKIP] Already exists in your Drive: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline
[COPY] /content/drive/MyDrive/Shared checkpoints/grpo_lab_checkpoints/grpo_better_binary_reward -> /content/drive/MyDrive/grpo_lab_checkpoints/grpo_better_binary_reward
[SKIP] Already exists in your Drive: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_has_number
[COPY] /content/drive/MyDrive/Shared checkpoints/grpo_lab_checkpoints/grpo_bad_format_only -> /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_format_only


In [ ]:
print("\nCurrent contents of DRIVE_ROOT:")
if os.path.exists(DRIVE_ROOT):
    for name in sorted(os.listdir(DRIVE_ROOT)):
        print("-", name)
else:
    print("DRIVE_ROOT does not exist yet.")


Current contents of DRIVE_ROOT:
- sft_only_baseline


In [ ]:
# Create folder for baseline run (pre-finetune sft) if it doesn't exist
SFT_BASELINE_DIR = os.path.join(DRIVE_ROOT, "sft_only_baseline")
os.makedirs(SFT_BASELINE_DIR, exist_ok=True)
print(f"SFT_BASELINE_DIR: {SFT_BASELINE_DIR}")

SFT_BASELINE_DIR: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline


### GRPO chat template
Since we're using a base model, we should set a chat template. You can make your own chat template as well!
1. DeepSeek uses `<think>` and `</think>`, but this is **not** necessary - you can customize it however you like!
2. A `system_prompt` is recommended to at least guide the model's responses.

In [ ]:
# We need to also load the tokenizer for dataset processing
if os.path.exists(os.path.join(SFT_BASELINE_DIR, "tokenizer_config.json")):
    TOKENIZER_SOURCE = SFT_BASELINE_DIR
else:
    TOKENIZER_SOURCE = BASE_MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_SOURCE)
print(f"Loaded tokenizer from: {TOKENIZER_SOURCE}")

Loaded tokenizer from: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline


In [ ]:
reasoning_start = "<start_working_out>" # Acts as <think>
reasoning_end   = "<end_working_out>"   # Acts as </think>
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>'

We create a simple chat template below. Notice `add_generation_prompt` includes prepending `<start_working_out>` to guide the model to start its reasoning process.

In [ ]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with out specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

Let's see how our chat template behaves on an example:

In [ ]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>What is 1+1?<start_working_out>I think it's 2.<end_working_out><SOLUTION>2</SOLUTION><|endoftext|>What is 2+2?<start_working_out>"

### Pre-fine-tuning (SFT) for formatting

We now use a subset of NVIDIA’s [Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning), filtered to include only high-quality DeepSeek R1 traces.

We use only about 59 examples to first **prime** the model, or lightly pre-fine-tune it, so that it learns the custom formatting required later for GRPO.

We use the resulting checkpoint as our **baseline checkpoint** with **no RL fine-tuning**.

Why this baseline matters:
- it shows what the model can do after learning the required response format through SFT
- it isolates the added effect of RL fine-tuning
- it gives us a fair comparison point against the later reward-based GRPO checkpoints

Before running this stage, the notebook checks the baseline folder:

- If a finished SFT baseline already exists, the notebook skips SFT and uses that baseline directly.
- If no finished baseline exists, but an SFT checkpoint is available, the notebook resumes SFT from the latest checkpoint.
- If neither is available, the notebook starts SFT from the original pretrained model.

This avoids unnecessary model loading and helps reduce GPU memory pressure in Colab.

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# Select only numbers
dataset = dataset.iloc[np.where(is_number)[0]]

dataset

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

data/cot-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

,expected_answer,problem,generated_solution
0,14,Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$...,"<think>\nOkay, let's see. I need to solve the ..."
6,-2,Find the value of the parameter $a$ for which ...,"<think>\nOkay, so I need to find the value of ..."
9,18,What is the sum of all real numbers $x$ for wh...,"<think>\nOkay, so I need to solve the equation..."
13,2,Evaluate the sum \(\sum_{n=1}^\infty \frac{\ph...,"<think>\nOkay, so I need to evaluate the infin..."
17,30,What is the largest positive integer that divi...,"<think>\nAlright, so I need to find the larges..."
...,...,...,...
19243,244,"Let \( p \), \( q \), and \( r \) be the disti...","<think>\nOkay, so I need to find the value of ..."
19245,1,A bug is on the $0$ of a number line. At any p...,"<think>\nOkay, so I have this problem where a ..."
19247,4,A bus left point X for point Y. Two hours late...,"<think>\nOkay, let's tackle this problem step ..."
19248,18,Each interior angle of a regular n-gon measure...,"<think>\nOkay, let's see. I need to find the n..."


We have to format the dataset to follow our GRPO style formatting:

In [ ]:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)

Check to see if it worked:

In [ ]:
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Given $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<start_working_out>Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 

Let's truncate the pre fine-tuning dataset to `max_seq_length/2` since we don't want too long reasoning traces.

Note this might take 2 minutes!

In [ ]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape

(59, 5)

We then tokenize the messages and convert it to a Hugging Face compatible dataset format:

In [ ]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

Dataset({
    features: ['expected_answer', 'problem', 'generated_solution', 'Messages', 'N', 'text', '__index_level_0__'],
    num_rows: 59
})

Let's now pre fine-tune the model so it follows our custom GRPO formatting!

In [ ]:
# Lab behavior
RESUME_SFT_IF_AVAILABLE = True
SFT_LAB_EXTRA_STEPS = 10
SFT_FRESH_MAX_STEPS = 100
SFT_SAVE_STEPS = 5

In [ ]:
SFT_LATEST_CKPT = get_latest_checkpoint(SFT_BASELINE_DIR) if RESUME_SFT_IF_AVAILABLE else None
SFT_START_STEP = get_checkpoint_step(SFT_LATEST_CKPT)
HAS_SFT_BASELINE = os.path.exists(os.path.join(SFT_BASELINE_DIR, "adapter_config.json"))

if HAS_SFT_BASELINE:
    SFT_STAGE_MODE = "skip_sft_use_existing_baseline"
    SFT_TARGET_MAX_STEPS = 0
    SFT_EFFECTIVE_SAVE_STEPS = 0
elif SFT_LATEST_CKPT is not None:
    SFT_TARGET_MAX_STEPS = SFT_START_STEP + SFT_LAB_EXTRA_STEPS
    SFT_EFFECTIVE_SAVE_STEPS = min(SFT_SAVE_STEPS, SFT_LAB_EXTRA_STEPS)
    SFT_STAGE_MODE = "resume_sft_from_checkpoint"
else:
    SFT_TARGET_MAX_STEPS = SFT_FRESH_MAX_STEPS
    SFT_EFFECTIVE_SAVE_STEPS = SFT_SAVE_STEPS
    SFT_STAGE_MODE = "fresh_sft_from_pretrained"

print("=" * 80)
print(f"SFT run mode: {SFT_STAGE_MODE}")
print(f"SFT latest checkpoint: {SFT_LATEST_CKPT}")
print(f"SFT start step: {SFT_START_STEP}")
print(f"SFT target max steps: {SFT_TARGET_MAX_STEPS}")
print(f"SFT save every: {SFT_EFFECTIVE_SAVE_STEPS} steps")
print("=" * 80)

SFT run mode: skip_sft_use_existing_baseline
SFT latest checkpoint: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline/checkpoint-110
SFT start step: 110
SFT target max steps: 0
SFT save every: 0 steps


In [ ]:
from trl import SFTTrainer, SFTConfig
sft_args = SFTConfig(
        output_dir = SFT_BASELINE_DIR,
        max_steps = SFT_TARGET_MAX_STEPS,
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
        save_steps=SFT_EFFECTIVE_SAVE_STEPS, # For checkpoints
        save_total_limit=2, # For checkpoints

    )

In [ ]:
if SFT_STAGE_MODE == "skip_sft_use_existing_baseline":
    print("Skipping SFT stage.")
    print(f"Using existing SFT baseline at: {SFT_BASELINE_DIR}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = SFT_BASELINE_DIR,
        max_seq_length = max_seq_length,
        load_in_4bit = False,
        fast_inference = False,
        gpu_memory_utilization = 0.9, # Reduce if out of memory
    )

elif SFT_STAGE_MODE == "resume_sft_from_checkpoint":
    print("Resuming SFT from the latest checkpoint.")
    print(f"Checkpoint: {SFT_LATEST_CKPT}")
    print(f"Baseline directory: {SFT_BASELINE_DIR}")

    # Load the model from the baseline folder, then resume optimizer/trainer
    # state from the latest checkpoint.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = SFT_BASELINE_DIR,
        max_seq_length = max_seq_length,
        load_in_4bit = False,
        fast_inference = False,  # Training mode, not fast inference mode
        gpu_memory_utilization = 0.9, # Reduce if out of memory
    )

    sft_trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = dataset,
        args = sft_args,
    )

    sft_trainer.train(resume_from_checkpoint = SFT_LATEST_CKPT)

    # After resuming and finishing, overwrite the baseline root with the latest SFT state.
    model.save_pretrained(SFT_BASELINE_DIR)
    tokenizer.save_pretrained(SFT_BASELINE_DIR)

else:
    print("No SFT baseline or checkpoint found. Starting fresh SFT.")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3-4B-Base",
        max_seq_length = max_seq_length,
        load_in_4bit = False, # False for LoRA 16bit
        fast_inference = True, # Enable vLLM fast inference
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.9, # Reduce if out of memory
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
        target_modules = [
                  "q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = lora_rank*2, # *2 speeds up training
        use_gradient_checkpointing = "unsloth", # Reduces memory usage
        random_state = 3407,
    )

    sft_trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = dataset,
        args = sft_args,
    )

    sft_trainer.train()

    # Save the finished SFT baseline to the root folder.
    model.save_pretrained(SFT_BASELINE_DIR)
    tokenizer.save_pretrained(SFT_BASELINE_DIR)

# Save or refresh baseline metadata after any successful SFT state
# (fresh run or resumed run). This keeps the baseline folder self-describing.
SFT_BASELINE_DESCRIPTION = (
    "Baseline checkpoint after supervised fine-tuning only, before any RL fine-tuning. "
    "Used to isolate the effect of reward-based GRPO training."
)

baseline_metadata = {
    "experiment_name": "sft_only_baseline",
    "label": "SFT-only baseline",
    "description": SFT_BASELINE_DESCRIPTION,
    "reward_functions": [],
}

with open(os.path.join(SFT_BASELINE_DIR, "experiment_metadata.json"), "w") as f:
    json.dump(baseline_metadata, f, indent=2)

print(f"Baseline metadata saved to: {os.path.join(SFT_BASELINE_DIR, 'experiment_metadata.json')}")

Skipping SFT stage.
Using existing SFT baseline at: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline
==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

Unsloth 2026.3.17 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Baseline metadata saved to: /content/drive/MyDrive/grpo_lab_checkpoints/sft_only_baseline/experiment_metadata.json


Let's check if the model has learnt to follow the custom format:

In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Jenifer has 82 cents in pennies and nickels. Her younger brother mistook all her nickels for dimes and counted the total as $1.47. How many pennies does Jenifer have?<start_working_out>Okay, let's see. Jenifer has 82 cents in pennies and nickels. Her brother thought all the nickels were dimes and counted the total as $1.47. I need to find out how many pennies she has.

First, let me convert everything to cents to make it easier. $1.47 is 147 cents. So, if the brother counted the nickels as dimes, that means each nickel is being counted as 10 cents instead of 5. So, the difference per nickel is 10 - 5 = 5 cents.

Let me denote the number of nickels as N. Since each nickel is overcounted by 5 cents, the total overcount is 5N cents. The actual total is 82 cents, but the brother's cou

Yes it did follow the formatting! Great! Let's remove some items before the GRPO step

In [ ]:
import gc
del dataset
gc.collect()
torch.cuda.empty_cache()

### Data Prep
<a name="Data"></a>

We're using Hugging Face's [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed). You can also utilize OpenAI's famous [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

README.md: 0.00B [00:00, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14116 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 14116
})

Let's look at the first row:

In [ ]:
dataset[0]["prompt"]

'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.'

In [ ]:
dataset[0]["solution"]

'34'

In GSM8K, we notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

In [ ]:
def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text
extract_hash_answer(dataset[0]["solution"])

'34'

Let's map the dataset! and see the first row:

In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

{'prompt': [{'content': 'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>',
   'role': 'system'},
  {'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.',
   'role': 'user'}],
 'solution': '34',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $

Get the top 90% prompt length so we don't accidentally truncate them!

Ie we'll remove the top 10% long prompts.

In [ ]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outside triangle $ABC$ such that $\angle BAD = \angle DAC$ and $\angle BDC = 90^\circ$. Suppose that $AD = 1$ and that $\frac{BD}{CD} = \frac{3}{2}$. If $AB + AC$ can be expressed in the form $\frac{a\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.<start_working_out>


Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

Max Length =  201


We create a regex format to match the reasoning sections and answers:

In [ ]:

# Add optional EOS token matching
solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"

match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
match_format

re.compile(r'<end_working_out>.*?<SOLUTION>(.+?)</SOLUTION>[\s]{0,}(?:<\|endoftext\|>)?[\s]{0,}$',
re.MULTILINE|re.DOTALL|re.UNICODE)

We verify it works:

In [ ]:
match_format.findall(
    "Let me think!<end_working_out>"\
    f"<SOLUTION>\n2\n</SOLUTION>",
)

['\n2\n']

In [ ]:
match_format.findall(
    "<start_working_out>Let me think!<end_working_out>"\
    f"<SOLUTION>  2  </SOLUTION>\n\n",
)

['  2  ']

### Weak reward baselines

We now define two deliberately weak reward functions.

These rewards are useful for teaching because they illustrate an important point:
a reward can be easy to compute, and still be poorly aligned with the actual task.

Both of these rewards can produce misleading learning signals:
- one rewards **structure without correctness**
- the other rewards **surface-level output without real task success**

We use them as weak baselines before moving to a slightly better reward based on answer correctness.

#### Bad reward 1: format-only reward

This reward checks only whether the response follows the expected structure.

Why this is bad:
- it rewards formatting, not correctness
- a beautifully formatted wrong answer can still receive a high score
- it can teach the model to imitate structure without solving the task

In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

#### Bad reward 2: has-a-number reward

This reward checks only whether the response contains a number.

Why this is bad:
- it is extremely easy to game
- it is only weakly related to solving the actual problem
- the model can receive reward without producing a correct or meaningful answer

In [ ]:
def reward_has_number(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        has_digit = any(ch.isdigit() for ch in response)
        scores.append(1.0 if has_digit else 0.0)
    return scores

### Slightly better reward: binary correctness

We now use a binary correctness reward.

In this setup, the model receives:
- **1** if the final answer is correct
- **0** if the final answer is incorrect

This reward is still simple and still limited, but it is more meaningfully aligned with the task than the previous weak reward baselines.

It therefore serves as a stronger starting point, while still leaving room for improvement in the final submission task.

In [ ]:
def check_answer_binary(prompts, completions, answer, **kwargs):
    """
    Binary correctness-based reward.

    Returns:
      1.0 if the extracted final answer matches the ground truth
      0.0 otherwise
    """
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1).strip()
        if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        true_answer = str(true_answer).strip()
        is_correct = (guess is not None) and (guess == true_answer)
        scores.append(1.0 if is_correct else 0.0)
    return scores

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

## Reward comparison

We now compare three GRPO reward:

1. **Bad reward 1**: format-only reward
2. **Bad reward 2**: has-a-number reward
3. **Slightly better**: binary correctness reward

This comparison helps us see that:
- some rewards are easy to compute but poorly aligned with the task
- even a simple improvement in alignment can change model behavior
- stronger reward design is something you will develop yourself in the submission task

#### Reward weights

We define a small helper function that preserves the reward-function interface expected by the GRPO trainer, while still letting us scale each reward if needed.

In this notebook, the reward weights are kept simple because the main goal is **not** to engineer the best possible reward. Instead, the goal is to compare a few deliberately weak baselines against one slightly better baseline.

So here, reward weighting is just a lightweight way to keep the experiment interface consistent, not a sophisticated reward-design strategy.

In [ ]:
def make_weighted_reward(func, weight: float):
    """
    Wraps a reward function so its outputs are scaled by `weight`.

    Original func signature is unchanged:
      func(prompts, completions, answers, **kwargs) -> List[float]
    """
    def wrapped(*args, **kwargs):
        scores = func(*args, **kwargs)
        return [weight * s for s in scores]
    wrapped.__name__ = f"{func.__name__}_w{weight}"
    return wrapped


In [ ]:
reward_weights = {
    "reward_format_only": 1.0,
    "reward_has_number": 1.0,
    "check_answer_binary": 1.0,
}

In [ ]:
#@title Select experiment { run: "auto" }
EXPERIMENT_NAME = "bad_has_number"  # @param ["bad_format_only", "bad_has_number", "better_binary_reward"]

experiment_configs = {
    "bad_format_only": {
        "label": "Bad reward baseline 1: format-only",
        "description": (
            "GRPO fine-tuning with a reward that checks only whether the output follows "
            "the expected format. This is intentionally weak because it can reward "
            "well-formatted but incorrect answers."
        ),
        "reward_funcs": [
            make_weighted_reward(
                match_format_exactly,
                reward_weights["reward_format_only"],
            ),
        ],
        "checkpoint_dir": os.path.join(DRIVE_ROOT, "grpo_bad_format_only"),
    },

    "bad_has_number": {
        "label": "Bad reward baseline 2: has-a-number",
        "description": (
            "GRPO fine-tuning with a reward that checks only whether the output contains "
            "a number. This is intentionally weak because it is easy to game and only "
            "weakly related to solving the task."
        ),
        "reward_funcs": [
            make_weighted_reward(
                reward_has_number,
                reward_weights["reward_has_number"],
            ),
        ],
        "checkpoint_dir": os.path.join(DRIVE_ROOT, "grpo_bad_has_number"),
    },

    "better_binary_reward": {
        "label": "Slightly better baseline: binary correctness",
        "description": (
            "GRPO fine-tuning with a binary correctness reward: 1 for a correct final "
            "answer and 0 otherwise. This is still simple and sparse, but it is more "
            "meaningfully aligned with the task than the two weak baselines."
        ),
        "reward_funcs": [
            make_weighted_reward(
                check_answer_binary,
                reward_weights["check_answer_binary"],
            ),
        ],
        "checkpoint_dir": os.path.join(DRIVE_ROOT, "grpo_better_binary_reward"),
    },
}

experiment = experiment_configs[EXPERIMENT_NAME]
reward_funcs = experiment["reward_funcs"]
CKPT_DIR = experiment["checkpoint_dir"]

print("=" * 80)
print(f"Experiment: {experiment['label']}")
print(f"Description: {experiment['description']}")
print(f"Checkpoint dir: {CKPT_DIR}")
print("Reward functions:", [f.__name__ for f in reward_funcs])
print("=" * 80)

Experiment: Bad reward baseline 2: has-a-number
Description: GRPO fine-tuning with a reward that checks only whether the output contains a number. This is intentionally weak because it is easy to game and only weakly related to solving the task.
Checkpoint dir: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_has_number
Reward functions: ['reward_has_number_w1.0']


In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)

experiment_metadata = {
    "experiment_name": EXPERIMENT_NAME,
    "label": experiment["label"],
    "description": experiment["description"],
    "reward_functions": [f.__name__ for f in reward_funcs],
}

with open(os.path.join(CKPT_DIR, "experiment_metadata.json"), "w") as f:
    json.dump(experiment_metadata, f, indent=2)

print(f"Saved experiment metadata to {os.path.join(CKPT_DIR, 'experiment_metadata.json')}")

Saved experiment metadata to /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_has_number/experiment_metadata.json


## Important: start each GRPO baseline from the correct checkpoint

For a fair comparison, each reward baseline must follow its **own training path**.

- If a GRPO checkpoint already exists for the selected baseline, **resume training from the latest available checkpoint**
- If no GRPO checkpoint exists yet, **start from the shared SFT-only baseline**

This is important because we do **not** want to train one reward baseline and then continue directly into a different one.

Instead, the workflow is:

1. save the SFT-only baseline once
2. choose a reward baseline
3. check whether a GRPO checkpoint already exists for that baseline
4. if it exists, resume training from that checkpoint
5. if it does not exist, load the SFT-only baseline and start a fresh GRPO run
6. save the new GRPO checkpoints back to Google Drive

This keeps the reward baselines comparable while also making the lab faster by allowing training to continue from previously prepared checkpoints.

In [ ]:
# How many extra GRPO steps you can run during the lab
LAB_EXTRA_STEPS = 5

# If no GRPO checkpoint exists yet, this is the fresh-run target
FRESH_RUN_MAX_STEPS = 100

# Whether to resume from the latest GRPO checkpoint in Drive if one exists
RESUME_GRPO_IF_AVAILABLE = True

# Save more frequently during the short in-lab continuation
LAB_SAVE_STEPS = 5

print("=" * 80)
print(f"LAB_EXTRA_STEPS = {LAB_EXTRA_STEPS}")
print(f"FRESH_RUN_MAX_STEPS = {FRESH_RUN_MAX_STEPS}")
print(f"RESUME_GRPO_IF_AVAILABLE = {RESUME_GRPO_IF_AVAILABLE}")
print(f"LAB_SAVE_STEPS = {LAB_SAVE_STEPS}")
print("=" * 80)

LAB_EXTRA_STEPS = 5
FRESH_RUN_MAX_STEPS = 100
RESUME_GRPO_IF_AVAILABLE = True
LAB_SAVE_STEPS = 5


In [ ]:

LATEST_CKPT = get_latest_checkpoint(CKPT_DIR) if RESUME_GRPO_IF_AVAILABLE else None
START_STEP = get_checkpoint_step(LATEST_CKPT)

if LATEST_CKPT is not None:
    TARGET_MAX_STEPS = START_STEP + LAB_EXTRA_STEPS
    EFFECTIVE_SAVE_STEPS = min(LAB_SAVE_STEPS, LAB_EXTRA_STEPS)
    RUN_MODE = "resume"
else:
    TARGET_MAX_STEPS = FRESH_RUN_MAX_STEPS
    EFFECTIVE_SAVE_STEPS = 25
    RUN_MODE = "fresh_start"

print("=" * 80)
print(f"Run mode: {RUN_MODE}")
print(f"Latest checkpoint: {LATEST_CKPT}")
print(f"Start step: {START_STEP}")
print(f"Target max steps: {TARGET_MAX_STEPS}")
print(f"Save every: {EFFECTIVE_SAVE_STEPS} steps")
print("=" * 80)

Run mode: fresh_start
Latest checkpoint: None
Start step: 0
Target max steps: 100
Save every: 25 steps


In [ ]:
max_prompt_length = maximum_length + 1
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 2,
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,

    # Dynamic training horizon:
    # - fresh run: full training budget
    # - resumed run: continue only a few extra steps for the lab
    max_steps = TARGET_MAX_STEPS,

    # Save more frequently during short resumed runs
    save_steps = EFFECTIVE_SAVE_STEPS,
    save_total_limit = 3,

    report_to = "none",
    output_dir = CKPT_DIR,
)

print("=" * 80)
print(f"GRPO output_dir: {CKPT_DIR}")
print(f"GRPO max_steps: {TARGET_MAX_STEPS}")
print(f"GRPO save_steps: {EFFECTIVE_SAVE_STEPS}")
print("=" * 80)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2
GRPO output_dir: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_has_number
GRPO max_steps: 100
GRPO save_steps: 25


And now let's run the trainer.

As training progresses, you will see a table of values including the `reward` column. The goal here is **not necessarily** to obtain a strong model with every reward baseline. In fact, two of the provided rewards are intentionally poor.

What to watch for:
- whether the reward increases at all
- whether the reward is easy to exploit
- whether higher reward actually corresponds to better answers
- how differently the model behaves under weak versus slightly better reward signals

This is part of the point of the lab: a reward can be easy to optimize, and still be a poor objective.

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = reward_funcs,
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)

print("=" * 80)
print(f"Starting GRPO run: {experiment['label']}")
print(f"Description: {experiment['description']}")
print(f"Run mode: {RUN_MODE}")
if LATEST_CKPT is not None:
    print(f"Resuming from checkpoint: {LATEST_CKPT}")
    print(f"Continuing from step {START_STEP} to step {TARGET_MAX_STEPS}")
else:
    print("No GRPO checkpoint found. Starting fresh from the SFT-only baseline.")
    print(f"Training to step {TARGET_MAX_STEPS}")
print("=" * 80)

trainer.train(
    resume_from_checkpoint = LATEST_CKPT if LATEST_CKPT is not None else None
)

Starting GRPO run: Bad reward baseline 2: has-a-number
Description: GRPO fine-tuning with a reward that checks only whether the output contains a number. This is intentionally weak because it is easy to game and only weakly related to solving the task.
Run mode: fresh_start
No GRPO checkpoint found. Starting fresh from the SFT-only baseline.
Training to step 100


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,709 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 32768}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_has_number_w1.0 / mean,rewards / reward_has_number_w1.0 / std
1,0.000100,1.000000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.144118,1.000000,0.000000
2,0.101700,1.000000,0.000000,1706.500000,1567.000000,1846.000000,0.500000,1567.000000,1567.000000,1567.000000,101.745575,1.000000,0.000000


KeyboardInterrupt: 

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

#### Save the final LoRA adapter for inference

During GRPO training, full Trainer checkpoints are already saved inside the experiment folder and can be used to resume training later.

Here we also export a final LoRA adapter for easy loading during inference and checkpoint comparison.

In [ ]:
FINAL_LORA_DIR = os.path.join(CKPT_DIR, "final_lora")
os.makedirs(FINAL_LORA_DIR, exist_ok=True)

model.save_pretrained(FINAL_LORA_DIR)
tokenizer.save_pretrained(FINAL_LORA_DIR)

del model
del tokenizer
del trainer
torch.cuda.empty_cache()

print("=" * 80)
print(f"Saved final LoRA adapter to: {FINAL_LORA_DIR}")
print(f"Experiment: {experiment['label']}")
print("=" * 80)

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

with safe_open(os.path.join(FINAL_LORA_DIR, "adapter_model.safetensors"), framework = "pt") as f:
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert (n_zeros.item() != tensor.numel())

print(f"Verified adapter weights in {FINAL_LORA_DIR}")

Verified adapter weights in /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_has_number/final_lora


#### Load and compare saved artifacts

There are **two kinds of saved artifacts**:

1. **Trainer checkpoints** such as `checkpoint-100/`
   - these are used to **resume GRPO training**

2. **Final LoRA exports** such as `final_lora/`
   - these are used for **inference and side-by-side comparison**

#### Checkpoints available for comparison

For inference, we compare the following saved models:

- **baseline**: SFT only, with no RL fine-tuning
- **bad_format_only**: GRPO trained with the format-only reward
- **bad_has_number**: GRPO trained with the has-a-number reward
- **better_binary_reward**: GRPO trained with the binary correctness reward

This lets us directly observe how different reward signals change model behavior.

#### What to do in this section

Run the same prompt through:
1. the **SFT-only baseline**
2. the **format-only GRPO model**
3. the **has-a-number GRPO model**
4. the **binary-correctness GRPO model**

Then compare:
- whether the final answer is correct
- whether the output follows the required structure
- whether the model seems to exploit weak rewards
- whether the binary correctness reward behaves better than the two weak baselines

The goal is to see how changing only the reward construction can lead to noticeably different outcomes, even when the rest of the training pipeline stays the same.

In [ ]:
CHECKPOINT_OPTIONS = {
    "baseline": {
        "path": os.path.join(DRIVE_ROOT, "sft_only_baseline"),
        "label": "SFT-only baseline",
        "description": "Model after supervised fine-tuning only; no RL fine-tuning applied.",
    },
    "bad_format_only": {
        "path": os.path.join(DRIVE_ROOT, "grpo_bad_format_only", "final_lora"),
        "label": "Bad reward baseline 1: format-only",
        "description": "GRPO model trained with a reward that checks only whether the response follows the expected format.",
    },
    "bad_has_number": {
        "path": os.path.join(DRIVE_ROOT, "grpo_bad_has_number", "final_lora"),
        "label": "Bad reward baseline 2: has-a-number",
        "description": "GRPO model trained with a reward that checks only whether the response contains a number.",
    },
    "better_binary_reward": {
        "path": os.path.join(DRIVE_ROOT, "grpo_better_binary_reward", "final_lora"),
        "label": "Slightly better baseline: binary correctness",
        "description": "GRPO model trained with a binary correctness reward.",
    },
}

CHECKPOINT_TO_LOAD = "bad_has_number"  # @param ["baseline", "bad_format_only", "bad_has_number", "better_binary_reward"]
selected = CHECKPOINT_OPTIONS[CHECKPOINT_TO_LOAD]

assert os.path.exists(selected["path"]), (
    f"Checkpoint not found at {selected['path']}. "
    "Make sure you trained and saved that experiment first."
)

print("=" * 80)
print(f"Loading: {selected['label']}")
print(f"Description: {selected['description']}")
print(f"Path: {selected['path']}")
print("=" * 80)

infer_model, infer_tokenizer = FastLanguageModel.from_pretrained(
    model_name = selected["path"],
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    fast_inference = False,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9,
)

#Optional Unsloth inference optimization
FastLanguageModel.for_inference(infer_model)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = infer_tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
)

_ = infer_model.generate(
    **infer_tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    top_k = 50,
    max_new_tokens = 2048,
    streamer = TextStreamer(infer_tokenizer, skip_prompt = False),
    )

# cleaning
del infer_model
del infer_tokenizer
torch.cuda.empty_cache()

Loading: Bad reward baseline 2: has-a-number
Description: GRPO model trained with a reward that checks only whether the response contains a number.
Path: /content/drive/MyDrive/grpo_lab_checkpoints/grpo_bad_has_number/final_lora
==((====))==  Unsloth 2026.3.15: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>What is the sqrt of 101?<start_working_out>Okay, so I need to find the square root of 101. Hmm, let me think. I remember that square roots of perfect squares are whole numbers, but 101 isn't a perfect square. The closest perfect squares around 101 are 100 and 121. The square root of 100 is 10, and the square root of 121 is 11. So, the square root of 101 must be between 10 and 11. 

But the problem is asking for the square root, not an approximation. Maybe I can express it in a different form? Like a decimal or a fraction? Let me try to calculate it. 

Wait, maybe I can use the fact that 101 is 1 more than 100. So, √101 = √(100 + 1). Hmm, but that doesn't directly help. Alternatively, perhaps I can use the binomial theorem or some approximation method. 

Wait, there's a formula for

## Week 9 Submission Task

In this lab, you fine-tuned Qwen3 4B for mathematical reasoning using GRPO with LoRA as the PEFT technique.

The notebook provided:
- **two deliberately weak reward baselines**
  - a **format-only reward**
  - a **has-a-number reward**
- and **one slightly better reward baseline**
  - a **binary correctness reward**

These were included to show that reward design matters: some rewards are easy to compute but poorly aligned with the actual task, while even a simple improvement in alignment can already change model behavior.

In this assessment, your task is to design and implement a **better reward function** that improves on these provided baselines and more effectively guides the model toward correct, well-structured mathematical outputs.

Extend the notebook by designing and implementing improved reward functions. Your reward should go beyond the provided baselines and aim to produce a model that performs better on the math reasoning task.

**What you need to do:**

1. Analyse the existing reward baselines provided in the notebook and explain their limitations.
2. Design one or more new reward functions that better capture what a good mathematical response should look like.
3. Implement your reward functions in the notebook and run the GRPO training pipeline with them.
4. Evaluate and compare the results against the provided baselines, using appropriate metrics and/or qualitative examples.

**Hints:**

A reward function is better if it provides a more informative and better-aligned training signal. Here are some questions to guide your thinking:

1. Does the model get the final answer correct? How can you check this reliably?
2. Can you give partial credit, rather than only binary correct/incorrect feedback?
3. How should the reward handle equivalent numerical answers or small formatting variations?
4. Can you encourage useful structure without rewarding superficial formatting alone?
5. How can you make the reward harder to game while still keeping it easy to compute?

The goal is not simply to make the reward more complicated, but to make it **better aligned with the behavior you actually want**.